<img src="../../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# EU government bond yields - data visualisation exercise - Solution

---

In this exercise we'll use a dataset of long term government bond yields from [data.europa.eu](https://data.europa.eu/euodp/en/data/dataset/JmCLAeHrvXI80AhjeV3zYQ).  IT is called `eu-govt-bonds-cleaned.csv`

### 1. Load the data

Use the version of the dataset you exported from the `pandas` exercise. Otherwise you can use a cleaned version provided in the `datasets` folder.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/eu-govt-bonds-cleaned.csv")
df.head()

### 2. Plot the average rate per country using a *horizontal bar chart*

Remember to get into the habit of labelling your axes, adding a title etc.

*Bonus: sort the data so that bars are in descending order in size (from top to bottom)*

In [ ]:
# as we saw in the pandas exercise, this "wide format" data means calculating average rate
# is not the usual group by country then aggregate, but instead we "squash" all the numeric values
# along the column axis

avg_by_country = (
    df
      .drop("country_code", axis=1) # no need for the code
      .dropna(subset=["Country"]) # drop missing countries (i.e. with non-country codes)
      .set_index("Country").mean(axis=1) # calculate mean along columns
)
avg_by_country.head()

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(figsize=(10, 6))

avg_by_country.sort_values().plot.barh(ax=ax)

ax.set(title="Avg. government bond yields Feb 2019 - Jan 2020",
       xlabel="Mean yield rate",
       ylabel="Country")

plt.show()

### 3. Plot each country's rates over time on a line chart

### 3. a) Get the data in the right format

To make this easier using `matplotlib`, getting the data in the right format is key, so this answer will take a few parts to get right.

The format we're aiming for is each column being a country and each row a month:

| Month | Austria   | ... |
|---------|---------|------|
| 2019M02 | 0.45 | ... |
| 2019M03 | 0.38 | ... |

The [relevant `pandas` documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/reshaping.html) should help you get the data into the desired format.

In [ ]:
# One approach is to simply transpose the table
# (making the country the index first so it becomes the column names)

df_for_time_series = (
    df
      .drop("country_code", axis=1) # redundant column
      .dropna(subset=["Country"]) # choosing to remove NULL countries
      .set_index("Country") # preserve country as column name when transposing
      .T # and transpose!
)
df_for_time_series.head()

In [ ]:
# The other option is a two-step stack and unstack
# first stack to make each row a country-date combination,
# then unstack to leave dates as rows, countries as columns
(
    df
      .drop("country_code", axis=1)
      .dropna(subset=["Country"])
      .set_index("Country") # again, preserve country names as future column names
      .stack() # data is now one row per country + month
      .unstack(0) # unstack the first column (country) to make it the new set of columns
)

### 3. b) Now plot the time series

Remember to include axis labels, a title, and a legend

*Bonus: replace the 2019M02-style labels with a more human-readable label for months*

In [ ]:
# let's first 'fix' the label issue at source
# we'll set each row's date to be an actual datetime (usually a good idea with time series)
import datetime

# datetime.datetime(year, month, day) creates a new date object,
# which here we populate using the first 4 characters (year) of the month string
# the last two (month) and a hard-coded 1, so all months convert to a date object
# representing the first day of that month

# using a list comprehension

dates = [datetime.datetime(int(i[:4]), # first 4 characters = year
                           int(i[-2:]), # last 2 = month
                           1 # hard-coded 1
                           ) for i in df_for_time_series.index]
dates

In [ ]:
# now use this to re-index the transposed dataframe

df_for_time_series.index = dates
df_for_time_series.head()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

df_for_time_series.plot(ax=ax)

ax.set(title="Government bond yields over time",
       xlabel="Date",
       ylabel="Rate")

plt.show()

### 3. c) Create a more interpretable visual

Clearly that's too many countries to put on a static image.

Recreate the visual but only for the following countries: Belgium, Germany, Denmark, Finland, and Slovakia

Add a faint, dashed, horizontal line at y=0 so it's clear where rates go above/below 0

In [ ]:
df_five_countries = df_for_time_series[["Belgium", "Germany", "Denmark", "Finland", "Slovakia"]]

fig, ax = plt.subplots(figsize=(16, 6))

df_five_countries.plot(ax=ax)

ax.set(title="Government bond yields over time (zoomed in to five countries)",
       xlabel="Date",
       ylabel="Rate")

ax.axhline(0, linestyle="dashed", color="black", alpha=0.5)

plt.show()

### 4. [Advanced] Use the `seaborn` library to visualise the spread of rates per country

Choose an appropriate visualisation to show not just the average, but the distribution of rate values per country. Your visualisation should show one row/column per country and the resulting spread/distribution of rate.

Seaborn has a few options for you to try, have a look at [the example gallery](https://seaborn.pydata.org/examples/index.html).

*Hint: you may find it easier to convert your data to a narrow format first*

Reminder:

The idea is rather than having a column per month, have a "month" column and have all the values stored in it. The data should therefore look like this:

| Country | Month   | Rate |
|---------|---------|------|
| Austria | 2019M02 | 0.45 |
| Austria | 2019M03 | 0.38 |

Use [this pandas article](https://pandas.pydata.org/pandas-docs/stable/user_guide/reshaping.html) to help you find the right command.

You can read more about wide vs. narrow data here: [https://en.wikipedia.org/wiki/Wide_and_narrow_data](https://en.wikipedia.org/wiki/Wide_and_narrow_data)

In [ ]:
df_stacked = (
    df
     .drop("country_code", axis=1)
     .dropna(subset=["Country"])
     .set_index("Country")
     .stack() # this puts all the month columns in one column
     .reset_index() # convert the resulting Series to a DataFrame
     .rename(columns={"level_1": "month", 0: "rate"}) # rename the resulting default column names
)

df_stacked.head()

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(3, 1, figsize=(16, 24))

# options include boxplot, swarmplot, violinplot
# let's show all three!

sns.boxplot(data=df_stacked,
            x="rate",
            y="Country",
            ax=ax[0])

ax[0].set_title("Box plot")

sns.swarmplot(data=df_stacked,
              x="rate",
              y="Country",
              ax=ax[1])
ax[1].set_title("Swarm plot")

sns.violinplot(data=df_stacked,
               x="rate",
               y="Country",
               ax=ax[2])
ax[2].set_title("Violin plot")

plt.show()